In [1]:
!nvidia-smi

Thu Oct  8 07:09:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from numba import cuda

device = cuda.get_current_device()
print(f"Device Name: {device.name}")
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Warp Size: {device.WARP_SIZE}")

Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count (SMs): 40
Max Threads Per Block: 1024
Warp Size: 32


In [4]:
import numpy as np
import time
from numba import cuda

# 1. CPU Reference Implementation
def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

# 2. GPU Kernel Implementation
@cuda.jit
def double_gpu_kernel(d_in, d_out):
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

In [5]:
def benchmark_run(N, student_id_seed=230103191):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)

    # CPU Pure Loop
    if N <= 1_000_000:
        t0 = time.perf_counter()
        double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan')

    # GPU Setup
    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    # Warmup compilation run
    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()

    # GPU Roundtrip Measurement
    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0

    # GPU Compute-Only Measurement
    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0

    return cpu_time_ms, gpu_total_ms, gpu_compute_ms


print(f"{'N':<12} | {'CPU (ms)':<12} | {'GPU Total (ms)':<15} | {'GPU Kernel (ms)':<15} | Winner")
print("-" * 75)

sizes = [100, 1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]
for N in sizes:
    cpu_ms, total_ms, kernel_ms = benchmark_run(N)
    if np.isnan(cpu_ms):
        winner = "GPU"
        cpu_str = "N/A (skip)"
    else:
        winner = "CPU" if cpu_ms < total_ms else "GPU"
        cpu_str = f"{cpu_ms:.4f}"
    print(f"{N:<12} | {cpu_str:<12} | {total_ms:<15.4f} | {kernel_ms:<15.4f} | {winner}")

N            | CPU (ms)     | GPU Total (ms)  | GPU Kernel (ms) | Winner
---------------------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


100          | 0.0560       | 1.4407          | 0.0852          | CPU
1000         | 0.2837       | 0.5651          | 0.0637          | CPU
10000        | 2.5870       | 0.5537          | 0.0597          | GPU
100000       | 25.9112      | 0.6639          | 0.0655          | GPU
1000000      | 281.5825     | 3.0983          | 0.1304          | GPU
5000000      | N/A (skip)   | 13.8746         | 0.2720          | GPU
10000000     | N/A (skip)   | 25.4073         | 0.6092          | GPU


In [6]:
from numba import cuda
import numpy as np

# DEFECTIVE KERNEL: Missing 'if idx < n' check!
@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    # FORBIDDEN: Writing without bounds check!
    d_arr[idx] = 999.0

# TEST HARNESS
N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

threads_per_block = 256
blocks_per_grid = 4  # 4 * 256 = 1024 threads, covering N=1000 + 24 extra

print("Launching defective kernel with 1024 threads on 1000-element array...")
defective_kernel[blocks_per_grid, threads_per_block](d_arr)
cuda.synchronize()
print("Execution finished (or did it crash?). Checking host copy...")
h_result = d_arr.copy_to_host()
print(f"Elements processed: {len(h_result)}. Last element: {h_result[-1]}")

Launching defective kernel with 1024 threads on 1000-element array...
Execution finished (or did it crash?). Checking host copy...
Elements processed: 1000. Last element: 999.0


In [7]:
from numba import cuda
import numpy as np

@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)

    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0

# Configuration
N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block  # Ceiling division

log_b = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_t = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_g = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_a = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)

inspect_threads_kernel[blocks_per_grid, threads_per_block](log_b, log_t, log_g, log_a, N)
cuda.synchronize()

print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<10}")
print("-" * 50)
for i in range(32, 40):  # Print the critical boundary transition (Block 4)
    status = "ACTIVE" if log_a[i] == 1 else "IDLE (GUARDED)"
    print(f"{log_g[i]:<10} | {log_b[i]:<10} | {log_t[i]:<10} | {status:<10}")

Global ID  | Block ID   | Thread ID  | Status    
--------------------------------------------------
32         | 4          | 0          | ACTIVE    
33         | 4          | 1          | ACTIVE    
34         | 4          | 2          | ACTIVE    
35         | 4          | 3          | ACTIVE    
36         | 4          | 4          | ACTIVE    
37         | 4          | 5          | IDLE (GUARDED)
38         | 4          | 6          | IDLE (GUARDED)
39         | 4          | 7          | IDLE (GUARDED)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [8]:
from numba import cuda
import numpy as np
import time

# BUGGY NAIVE SUMMATION (Race Condition)
@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        # DATA RACE: Read-Modify-Write without lock
        d_total[0] += d_arr[idx]

# CORRECT ATOMIC SUMMATION
@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        # HARDWARE ATOMIC LOCK: Serializes updates
        cuda.atomic.add(d_total, 0, d_arr[idx])

In [9]:
N = 50_000
ones = np.ones(N, dtype=np.float32)

threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

print("--- Running Naive Kernel (5 Trials) ---")
for trial in range(1, 6):
    d_arr = cuda.to_device(ones)
    d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
    naive_sum_kernel[blocks_per_grid, threads_per_block](d_arr, d_total)
    cuda.synchronize()
    result = d_total.copy_to_host()[0]
    lost = 50000.0 - result
    print(f"Trial {trial}: Naive Sum = {result:.1f} | Expected = 50000.0 | Lost Updates = {lost:.1f}")

print("\n--- Running Atomic Kernel ---")
# Warmup atomic
d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
d_total_warm = cuda.to_device(np.zeros(1, dtype=np.float32))
atomic_sum_kernel[1, 32](d_warmup, d_total_warm)
cuda.synchronize()

# Atomic timing and result
d_arr = cuda.to_device(ones)
d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
t0 = time.perf_counter()
atomic_sum_kernel[blocks_per_grid, threads_per_block](d_arr, d_total)
cuda.synchronize()
atomic_time_ms = (time.perf_counter() - t0) * 1000.0
atomic_result = d_total.copy_to_host()[0]
print(f"Atomic Sum = {atomic_result:.1f} | Expected = 50000.0 | Time = {atomic_time_ms:.4f} ms")

# Naive timing for comparison
d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
t0 = time.perf_counter()
naive_sum_kernel[blocks_per_grid, threads_per_block](d_arr, d_total)
cuda.synchronize()
naive_time_ms = (time.perf_counter() - t0) * 1000.0
print(f"Naive Time = {naive_time_ms:.4f} ms")

--- Running Naive Kernel (5 Trials) ---
Trial 1: Naive Sum = 2.0 | Expected = 50000.0 | Lost Updates = 49998.0
Trial 2: Naive Sum = 2.0 | Expected = 50000.0 | Lost Updates = 49998.0
Trial 3: Naive Sum = 2.0 | Expected = 50000.0 | Lost Updates = 49998.0
Trial 4: Naive Sum = 2.0 | Expected = 50000.0 | Lost Updates = 49998.0
Trial 5: Naive Sum = 2.0 | Expected = 50000.0 | Lost Updates = 49998.0

--- Running Atomic Kernel ---
Atomic Sum = 50000.0 | Expected = 50000.0 | Time = 0.6681 ms
Naive Time = 0.4988 ms


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [10]:
import math
from numba import cuda
import numpy as np

@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    x, y = cuda.grid(2)
    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)
        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))
        intensity = 1.0 - (dist / max_dist)
        if intensity < 0.0:
            intensity = 0.0
        d_canvas[y, x] = intensity

# 2D EXECUTION CONFIGURATION
WIDTH, HEIGHT = 1024, 1024
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)

# 2D Blocks (16x16 = 256 threads per block)
threads_2d = (16, 16)
blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1]
)

radial_vignette_2d[blocks_2d, threads_2d](d_canvas, WIDTH, HEIGHT, WIDTH // 2, HEIGHT // 2)
cuda.synchronize()

result_canvas = d_canvas.copy_to_host()
print(f"Generated 2D matrix shape: {result_canvas.shape}")
print(f"Center value at (512, 512): {result_canvas[512, 512]:.3f}")
print(f"Outer corner value at (0, 0): {result_canvas[0, 0]:.3f}")

Generated 2D matrix shape: (1024, 1024)
Center value at (512, 512): 1.000
Outer corner value at (0, 0): 0.000


In [12]:
import hashlib
from numba import cuda

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode('utf-8'))
    h.update(cuda.get_current_device().name.encode('utf-8'))
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(exp_table_crossover).encode('utf-8'))
    digest = h.hexdigest()[:16].upper()
    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)
    return digest

STUDENT_ID = "230103191"
CROSSOVER_N = 10000

generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

OFFICIAL VERIFICATION CHECKSUM TOKEN: 40DE3885EC1A615B


'40DE3885EC1A615B'